### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="anneal",
    dataset_year="1990",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/3/annealing",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/anneal/ && wget -P local-data-warehouse/anneal/ https://archive.ics.uci.edu/static/public/3/annealing.zip && unzip local-data-warehouse/anneal/annealing.zip -d local-data-warehouse/anneal/
""",
    # References
    academic_reference_bibtex="""@misc{uci1990annealing,
  title        = {Annealing},
  year         = {1990},
  author       = {Unknown},
  howpublished = {UCI Machine Learning Repository},
  url = {https://doi.org/10.24432/C5RW2F},
}
""",
    academic_reference_bibtex_key="uci1990annealing",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
- We combined train and test data into a single dataset.
- We replaced '?' with 'not_applicable' as described in the metadata.
- We renamed some features to remove '/' characters.
- Anomaly: In the original data, class 4 is in the metadata, but there are no samples in this class.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="classes",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="classes",
)

## Preprocessing

In [ ]:
import pandas as pd

data = pd.read_csv(f"{dataset_mold.path}/anneal.data", header=None)
test_data = pd.read_csv(f"{dataset_mold.path}/anneal.test", header=None)

# Concatenate the two datasets
df = pd.concat([data, test_data], ignore_index=True)
feature_names = [
    "family",
    "product-type",
    "steel",
    "carbon",
    "hardness",
    "temper_rolling",
    "condition",
    "formability",
    "strength",
    "non-ageing",
    "surface-finish",
    "surface-quality",
    "enamelability",
    "bc",
    "bf",
    "bt",
    "bw_me",  # original: "bw/me"
    "bl",
    "m",
    "chrom",
    "phos",
    "cbond",
    "marvi",
    "exptl",
    "ferro",
    "corr",
    "blue_bright_varn_clean",  # original: "blue/bright/varnish/clean"
    "lustre",
    "jurofm",
    "s",
    "p",
    "shape",
    "thick",
    "width",
    "len",
    "oil",
    "bore",
    "packing",
    "classes",
]

cat_features = [
  "family",
  "product-type",
  "steel",
  "temper_rolling",
  "condition", 
  "formability",
  "non-ageing",
  "surface-finish",
  "surface-quality",
  "enamelability",
  "bc",
  "bf",
  "bt",
  "bw_me",
  "bl",
  "m",
  "chrom",
  "phos",
  "cbond",
  "marvi",
  "exptl",
  "ferro",
  "corr",
  "blue_bright_varn_clean",
  "lustre",
  "jurofm",
  "s",
  "p",
  "shape",
  "oil",
  "bore",
  "packing"
]

df.columns = feature_names

df = df.replace("?", "not_applicable")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category") # NOTE: Unsure whether .astype("category") on the whole data is correct, since it defines cat codes using test data which is technically a leak


In [ ]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [ ]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)